# Tutorial Gaussian Mixture Models & EM Algorithm
## Probabilistic Clustering When K-Means Is Not Enough


---

### Learning Objectives
1. Derive the GMM generative model and log-likelihood.
2. Implement the E-step (responsibilities) and M-step (parameter updates) from scratch.
3. Prove EM monotonically increases log-likelihood.
4. Apply GMM to real UCI Iris data and compare with K-means.
5. Use BIC for model selection (choosing K).

### Accessibility Note
All figures use the colourblind-safe Burgundy/Sage Green palette. Scatter plots use distinct marker shapes. Bar charts have hatch patterns. Alt-text printed below every figure. H1→H2 heading hierarchy for screen readers.

## Imports and Setup

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from scipy.stats import multivariate_normal
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import warnings; warnings.filterwarnings('ignore')

# Burgundy / Sage Green
BURG   = '#722F37'
SAGE   = '#7FB069'
LBLURG = '#C4717A'
LSAGE  = '#B8D4A8'
XBURG  = '#F9ECED'
XSAGE  = '#EEF6E9'
CREAM  = '#FAFAF8'
DARK   = '#1A0F0F'
MID    = '#5C4A4A'
LGREY  = '#E4DDD8'
GOLD   = '#C9A84C'

plt.rcParams.update({
    'font.family':'DejaVu Sans','figure.facecolor':CREAM,
    'axes.facecolor':CREAM,'axes.spines.top':False,'axes.spines.right':False,
    'axes.linewidth':0.8,
})
np.random.seed(42)
print('Setup complete. Theme: Burgundy / Sage Green')

---
## 1 The GMM Model and EM Algorithm

The GMM density is:

$$p(x) = \\sum_{k=1}^{K} \\pi_k \\, \\mathcal{N}(x \\mid \\mu_k, \\Sigma_k)$$

**E-step** (given parameters, compute soft responsibilities):
$$r_{nk} = \\frac{\\pi_k \\mathcal{N}(x_n|\\mu_k,\\Sigma_k)}{\\sum_j \\pi_j \\mathcal{N}(x_n|\\mu_j,\\Sigma_j)}$$

**M-step** (given responsibilities, update parameters):
$$\\mu_k = \\frac{\\sum_n r_{nk} x_n}{N_k}, \\quad \\Sigma_k = \\frac{\\sum_n r_{nk}(x_n-\\mu_k)(x_n-\\mu_k)^\\top}{N_k}, \\quad \\pi_k = \\frac{N_k}{N}$$

In [ ]:
# From-scratch EM implementation
def e_step(X, means, covs, pis):
    """E-step: compute responsibilities r[n,k] for each point and component."""
    N, K = len(X), len(means)
    R = np.zeros((N, K))
    for k in range(K):
        R[:, k] = pis[k] * multivariate_normal.pdf(X, means[k], covs[k])
    denom = R.sum(axis=1, keepdims=True)
    R = R / (denom + 1e-300)  # normalise: each row sums to 1
    return R

def m_step(X, R):
    """M-step: update means, covariances, and mixing weights."""
    N, K = len(X), R.shape[1]
    Nk     = R.sum(axis=0)                          # effective counts
    means  = [(R[:,k:k+1] * X).sum(0) / Nk[k] for k in range(K)]
    covs   = [((R[:,k:k+1] * (X - means[k])).T @ (X - means[k])) / Nk[k]
              + np.eye(X.shape[1]) * 1e-6           # regularisation
              for k in range(K)]
    pis    = Nk / N
    return means, covs, pis.tolist()

def log_likelihood(X, means, covs, pis):
    """Compute the GMM log-likelihood (scalar)."""
    N, K = len(X), len(means)
    L = np.zeros((N, K))
    for k in range(K):
        L[:, k] = pis[k] * multivariate_normal.pdf(X, means[k], covs[k])
    return np.log(L.sum(axis=1) + 1e-300).sum()

def fit_gmm(X, K, n_iter=50, tol=1e-6, seed=42):
    """Fit a GMM using EM. Returns (means, covs, pis, responsibilities, ll_history)."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), K, replace=False)
    means = [X[i].copy() for i in idx]
    covs  = [np.eye(X.shape[1]) * 0.5 for _ in range(K)]
    pis   = [1.0/K] * K
    ll_hist = [log_likelihood(X, means, covs, pis)]
    for it in range(n_iter):
        R = e_step(X, means, covs, pis)
        means, covs, pis = m_step(X, R)
        ll = log_likelihood(X, means, covs, pis)
        ll_hist.append(ll)
        if abs(ll - ll_hist[-2]) < tol: break
    return means, covs, pis, R, ll_hist

# Generate synthetic 3-component data
means_true  = [np.array([-2., 1.5]), np.array([2., 2.]), np.array([0., -2.5])]
covs_true   = [np.array([[1.2,0.7],[0.7,0.8]]),
               np.array([[0.6,-0.3],[-0.3,1.0]]),
               np.array([[1.0,0.],[0.,0.4]])]
weights_true= [0.35, 0.35, 0.30]
all_pts=[]
for m,c,w in zip(means_true,covs_true,weights_true):
    all_pts.append(np.random.multivariate_normal(m,c,int(300*w)))
X_synth = np.vstack(all_pts)
idx=np.random.permutation(len(X_synth)); X_synth=X_synth[idx]

means_fit, covs_fit, pis_fit, R_fit, ll_hist = fit_gmm(X_synth, K=3)
print(f'Converged in {len(ll_hist)} iterations')
print(f'Final log-likelihood: {ll_hist[-1]:.2f}')
print(f'Mixing weights: {[f"{p:.3f}" for p in pis_fit]}')

In [ ]:
# EM in Action
def draw_ellipse(ax, mean, cov, color, alpha=0.18, lw=2.0, n_std=2.0):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]; vals, vecs = vals[order], vecs[:,order]
    theta = np.degrees(np.arctan2(*vecs[:,0][::-1]))
    w, h = 2*n_std*np.sqrt(vals)
    for a,lw2 in [(alpha,'none'),(0,lw)]:
        e = Ellipse(mean, width=w, height=h, angle=theta,
                    facecolor=color if a else 'none',
                    alpha=a if a else 1, edgecolor=color, linewidth=lw2, zorder=2)
        ax.add_patch(e)

colors3 = [BURG, SAGE, GOLD]; markers3 = ['o','s','^']
fig, axes = plt.subplots(2,2,figsize=(13,9.5),facecolor=CREAM)
fig.suptitle('Figure 1 EM Algorithm in Action: Initialisation to Convergence',
             fontsize=12.5,fontweight='bold',color=DARK,y=0.98)

mn0=[np.array([-3.,3.]),np.array([3.,0.5]),np.array([0.5,-3.5])]
cv0=[np.eye(2)*1.5]*3; pi0=[1/3]*3
R1=e_step(X_synth,mn0,cv0,pi0)
mn5,cv5,pi5=m_step(X_synth,R1); R5=e_step(X_synth,mn5,cv5,pi5)
for _ in range(4):
    mn5,cv5,pi5=m_step(X_synth,R5); R5=e_step(X_synth,mn5,cv5,pi5)
mnC,cvC,piC,RC,_=fit_gmm(X_synth,3)

stage_data=[
    ('A  |  Initialisation: Random Means',mn0,cv0,None),
    ('B  |  After E-Step: Soft Responsibilities',mn0,cv0,R1),
    ('C  |  After 5 Iterations',mn5,cv5,R5),
    ('D  |  Converged: Fitted ≈ True',mnC,cvC,RC),
]
for ax,(title,mn,cv,R) in zip(axes.flat,stage_data):
    ax.set_facecolor(CREAM)
    ax.set_title(title,fontsize=10,fontweight='bold',color=DARK,loc='left',pad=4)
    if R is not None:
        hard=R.argmax(1)
        for k,(col,mk) in enumerate(zip(colors3,markers3)):
            mask=hard==k
            ax.scatter(X_synth[mask,0],X_synth[mask,1],s=22,marker=mk,color=col,alpha=0.70,edgecolors='none',zorder=3)
    else:
        ax.scatter(X_synth[:,0],X_synth[:,1],s=22,color=LGREY,alpha=0.60,edgecolors='none',zorder=3)
    for k,(col,mk) in enumerate(zip(colors3,markers3)):
        draw_ellipse(ax,mn[k],cv[k],col,0.15,2.0)
        ax.scatter([mn[k][0]],[mn[k][1]],s=90,marker='X',color=col,edgecolors=DARK,linewidths=1.2,zorder=5)
    ax.set_xlabel('x₁',fontsize=9,color=DARK); ax.set_ylabel('x₂',fontsize=9,color=DARK)
    ax.set_xlim(-6,6); ax.set_ylim(-5.5,5.5)
    ax.spines['left'].set_color(LGREY); ax.spines['bottom'].set_color(LGREY)
plt.tight_layout(pad=1.2)
plt.savefig('fig1_em_in_action.png',dpi=150,bbox_inches='tight',facecolor=CREAM)
plt.show()
print('\n[ALT-TEXT] Figure 1: 2x2 grid showing EM progression. Panel A: grey points, random initial means (X markers). Panel B: after one E-step, points coloured burgundy/sage/gold using circle/square/triangle markers by dominant responsibility. Panel C: after 5 iterations, ellipses moving toward clusters. Panel D: converged - confidence ellipses match true generating distributions. Colourblind-safe burgundy/sage/gold palette, distinct marker shapes.')

In [ ]:
# Convergence + BIC + Iris
iris   = load_iris(); X_iris = StandardScaler().fit_transform(iris.data)
X_pca  = PCA(n_components=2).fit_transform(X_iris); y_iris = iris.target
iters  = np.arange(len(ll_hist))

fig=plt.figure(figsize=(13,4.2),facecolor=CREAM)
gs=GridSpec(1,3,figure=fig,wspace=0.42,left=0.07,right=0.97,top=0.84,bottom=0.18)
fig.suptitle('Figure 2  EM Convergence, BIC/AIC Model Selection, and GMM on Iris',
             fontsize=12,fontweight='bold',color=DARK,y=0.97)

ax1=fig.add_subplot(gs[0]); ax1.set_facecolor(CREAM)
ax1.plot(iters,ll_hist,color=BURG,lw=2.5,marker='o',markersize=3.5,markevery=3)
ax1.axhline(ll_hist[-1],color=LGREY,lw=1.2,linestyle=':')
ax1.text(len(iters)*0.9,ll_hist[-1]+8,'Converged',fontsize=7.5,color=MID,ha='right',style='italic')
ax1.fill_between(iters,ll_hist,ll_hist[-1],alpha=0.08,color=BURG)
ax1.set_xlabel('EM Iteration',fontsize=10,color=DARK)
ax1.set_ylabel('Log-likelihood',fontsize=10,color=DARK)
ax1.set_title('A  |  EM Convergence\n(monotone increase)',fontsize=9.5,fontweight='bold',color=DARK,loc='left')
ax1.spines['left'].set_color(LGREY); ax1.spines['bottom'].set_color(LGREY)

ax2=fig.add_subplot(gs[1]); ax2.set_facecolor(CREAM)
Ks=range(1,9)
bics=[GaussianMixture(k,n_init=5,random_state=42).fit(X_iris).bic(X_iris) for k in Ks]
aics=[GaussianMixture(k,n_init=5,random_state=42).fit(X_iris).aic(X_iris) for k in Ks]
ax2.plot(list(Ks),bics,color=BURG,lw=2.5,marker='s',markersize=6,label='BIC')
ax2.plot(list(Ks),aics,color=SAGE,lw=2.2,marker='^',markersize=6,linestyle='--',label='AIC')
best_k=Ks[np.argmin(bics)]; ax2.axvline(best_k,color=GOLD,lw=2.0,linestyle=':')
ax2.text(best_k+0.15,max(bics)*0.97,f'K={best_k}\n(BIC min)',fontsize=8,color=GOLD,fontweight='bold')
ax2.set_xlabel('Number of Components K',fontsize=10,color=DARK)
ax2.set_ylabel('Information Criterion',fontsize=9,color=DARK)
ax2.set_title('B  |  Model Selection: BIC/AIC\nSelects K=3 (true Iris species)',fontsize=9.5,fontweight='bold',color=DARK,loc='left')
ax2.legend(fontsize=8.5,frameon=False)
ax2.spines['left'].set_color(LGREY); ax2.spines['bottom'].set_color(LGREY)

ax3=fig.add_subplot(gs[2]); ax3.set_facecolor(CREAM)
gmm3=GaussianMixture(3,covariance_type='full',n_init=10,random_state=42)
gmm3.fit(X_iris); lbl=gmm3.predict(X_iris)
acc=(lbl==y_iris).mean()
for k,(col,mk,nm) in enumerate(zip([BURG,SAGE,GOLD],['o','s','^'],['Setosa','Versicolor','Virginica'])):
    mask=lbl==k
    ax3.scatter(X_pca[mask,0],X_pca[mask,1],s=28,marker=mk,color=col,
                alpha=0.75,edgecolors=DARK,linewidths=0.4,zorder=3,label=nm)
ax3.set_xlabel('PC1',fontsize=10,color=DARK); ax3.set_ylabel('PC2',fontsize=10,color=DARK)
ax3.set_title(f'C  |  GMM on Iris (PCA 2D)\nAccuracy ≈ {acc*100:.0f}%',fontsize=9.5,fontweight='bold',color=DARK,loc='left')
ax3.legend(fontsize=8,frameon=False)
ax3.spines['left'].set_color(LGREY); ax3.spines['bottom'].set_color(LGREY)

plt.savefig('fig2_convergence_bic_iris.png',dpi=160,bbox_inches='tight',facecolor=CREAM)
plt.show()
print('\n[ALT-TEXT] Figure 2: Three panels. Left (A): log-likelihood vs iteration in burgundy, rising steeply then converging at a dotted line. Centre (B): BIC (burgundy squares) and AIC (sage triangles) vs K; both minimise at K=3 marked by gold vertical line. Right (C): Iris PCA scatter with three clusters in burgundy circles (Setosa), sage squares (Versicolor), gold triangles (Virginica). Colourblind-safe palette, distinct marker shapes.')

In [ ]:
# GMM vs K-Means failure cases
fig=plt.figure(figsize=(13,4.5),facecolor=CREAM)
gs=GridSpec(1,3,figure=fig,wspace=0.38,left=0.06,right=0.97,top=0.84,bottom=0.16)
fig.suptitle('Figure 3  GMM vs K-Means: Elongated, Overlapping, Unequal-Density Clusters',
             fontsize=12,fontweight='bold',color=DARK,y=0.97)

# Case 1: Elongated
np.random.seed(10)
X_el=np.vstack([np.random.multivariate_normal([-3,0],[[2.5,0],[0,0.15]],80),
                 np.random.multivariate_normal([3,0], [[2.5,0],[0,0.15]],80)])
ax1=fig.add_subplot(gs[0]); ax1.set_facecolor(CREAM)
for k,(col,mk) in enumerate(zip([BURG,SAGE],['o','s'])):
    ax1.scatter(X_el[k*80:(k+1)*80,0],X_el[k*80:(k+1)*80,1],s=25,marker=mk,color=col,alpha=0.65,edgecolors='none')
ax1.axvline(0,color=LGREY,lw=1.5,linestyle='--',alpha=0.7)
ax1.text(0,1.2,'K-means\nboundary\n(wrong)',ha='center',fontsize=8,color=MID,style='italic')
ax1.set_title('A  |  Elongated Clusters\nK-means fails (non-spherical)',fontsize=9.5,fontweight='bold',color=DARK,loc='left')
ax1.set_xlabel('x₁',fontsize=9,color=DARK); ax1.set_ylabel('x₂',fontsize=9,color=DARK)
ax1.spines['left'].set_color(LGREY); ax1.spines['bottom'].set_color(LGREY)

# Case 2: Overlapping
np.random.seed(20)
X_ov=np.vstack([np.random.multivariate_normal([0,0],np.eye(2)*0.8,80),
                 np.random.multivariate_normal([1.5,0],np.eye(2)*0.8,80)])
ax2=fig.add_subplot(gs[1]); ax2.set_facecolor(CREAM)
gmm2=GaussianMixture(2,covariance_type='full',n_init=5,random_state=42)
gmm2.fit(X_ov); probs=gmm2.predict_proba(X_ov)
sc=ax2.scatter(X_ov[:,0],X_ov[:,1],s=28,c=probs[:,0],cmap='RdYlGn_r',
               edgecolors=DARK,linewidths=0.4,alpha=0.85,vmin=0,vmax=1)
plt.colorbar(sc,ax=ax2,label='P(Component 1)',fraction=0.04,pad=0.04)
ax2.set_title('B  |  Overlapping Clusters\nGMM: soft responsibilities',fontsize=9.5,fontweight='bold',color=DARK,loc='left')
ax2.set_xlabel('x₁',fontsize=9,color=DARK); ax2.set_ylabel('x₂',fontsize=9,color=DARK)
ax2.spines['left'].set_color(LGREY); ax2.spines['bottom'].set_color(LGREY)

# Case 3: Unequal density
np.random.seed(30)
X_un=np.vstack([np.random.multivariate_normal([-2,0],np.eye(2)*0.3,20),
                 np.random.multivariate_normal([2,0],np.eye(2)*1.5,120)])
ax3=fig.add_subplot(gs[2]); ax3.set_facecolor(CREAM)
gmm_un=GaussianMixture(2,covariance_type='full',n_init=5,random_state=42)
gmm_un.fit(X_un); lbl_un=gmm_un.predict(X_un)
for k,(col,mk) in enumerate(zip([BURG,SAGE],['o','s'])):
    mask=lbl_un==k
    ax3.scatter(X_un[mask,0],X_un[mask,1],s=28,marker=mk,color=col,
                alpha=0.75,edgecolors=DARK,linewidths=0.4,label=f'Cluster {k+1}')
    draw_ellipse(ax3,gmm_un.means_[k],gmm_un.covariances_[k],col,0.15,2.0)
ax3.set_title('C  |  Unequal Density\nGMM handles different sizes',fontsize=9.5,fontweight='bold',color=DARK,loc='left')
ax3.set_xlabel('x₁',fontsize=9,color=DARK); ax3.set_ylabel('x₂',fontsize=9,color=DARK)
ax3.legend(fontsize=8,frameon=False)
ax3.spines['left'].set_color(LGREY); ax3.spines['bottom'].set_color(LGREY)

plt.savefig('fig3_gmm_vs_kmeans.png',dpi=160,bbox_inches='tight',facecolor=CREAM)
plt.show()
print('\n[ALT-TEXT] Figure 3: Three-panel figure. Left (A): elongated clusters in burgundy circles and sage squares, split by an incorrect dashed vertical K-means boundary. Centre (B): overlapping clusters with RdYlGn_r gradient colourmap showing GMM soft responsibilities; colourbar from 0 to 1. Right (C): unequal-density clusters with confidence ellipses in burgundy and sage, showing GMM correctly captures a small dense cluster and a large diffuse cluster. Distinct marker shapes throughout.')

In [ ]:
# E-step responsibilities + M-step
R_conv=e_step(X_synth,mnC,cvC,piC)
sample_idx=np.sort(np.random.choice(len(X_synth),40,replace=False))
R_sample=R_conv[sample_idx]
pts_sorted=np.argsort(R_sample.argmax(1))
R_disp=R_sample[pts_sorted]

fig=plt.figure(figsize=(13,4.8),facecolor=CREAM)
gs=GridSpec(1,2,figure=fig,wspace=0.42,left=0.06,right=0.97,top=0.84,bottom=0.14)
fig.suptitle('Figure 4  E-Step Responsibilities and M-Step Parameter Updates',
             fontsize=12,fontweight='bold',color=DARK,y=0.97)

ax1=fig.add_subplot(gs[0]); ax1.set_facecolor(CREAM)
bottom=np.zeros(40)
for k,(col,hatch) in enumerate(zip([BURG,SAGE,GOLD],['//', 'xx', '..'])):
    ax1.bar(np.arange(40),R_disp[:,k],bottom=bottom,color=col,hatch=hatch,
            edgecolor='white',lw=0.5,alpha=0.88,label=f'r_{{n{k+1}}} Component {k+1}')
    bottom+=R_disp[:,k]
ax1.set_xlabel('Data point (sorted by dominant component)',fontsize=10,color=DARK)
ax1.set_ylabel('Responsibility  r_nk  in [0,1]',fontsize=10,color=DARK)
ax1.set_title('A  |  E-Step: Soft Responsibilities Per Point\n(each column sums to 1)',fontsize=9.5,fontweight='bold',color=DARK,loc='left')
ax1.legend(fontsize=8.5,frameon=False); ax1.set_xlim(-0.5,39.5); ax1.set_ylim(0,1.05)
ax1.spines['left'].set_color(LGREY); ax1.spines['bottom'].set_color(LGREY)

ax2=fig.add_subplot(gs[1]); ax2.set_facecolor(CREAM)
for k,(col,mk) in enumerate(zip(colors3,markers3)):
    w_k=R_conv[:,k]
    ax2.scatter(X_synth[:,0],X_synth[:,1],s=w_k*60+2,marker=mk,color=col,alpha=0.50,edgecolors='none')
    draw_ellipse(ax2,mnC[k],cvC[k],col,0.18,2.2)
    ax2.scatter([mnC[k][0]],[mnC[k][1]],s=100,marker='X',color=col,edgecolors=DARK,linewidths=1.3,zorder=5)
ax2.set_xlabel('x₁',fontsize=10,color=DARK); ax2.set_ylabel('x₂',fontsize=10,color=DARK)
ax2.set_title('B  |  M-Step: Weighted Re-fitting\n(point size ∝ responsibility)',fontsize=9.5,fontweight='bold',color=DARK,loc='left')
ax2.set_xlim(-6,6); ax2.set_ylim(-5.5,5.5)
ax2.spines['left'].set_color(LGREY); ax2.spines['bottom'].set_color(LGREY)

plt.savefig('fig4_estep_mstep.png',dpi=160,bbox_inches='tight',facecolor=CREAM)
plt.show()
print('\n[ALT-TEXT] Figure 4: Two-panel figure. Left (A): stacked bar chart of 40 sampled data points sorted by dominant component. Three stacked sections per bar in burgundy (forward-slash hatch), sage (cross hatch), gold (dot hatch). Mixed-responsibility points show multi-colour bars. Right (B): 300-point scatter with point sizes proportional to responsibility. Three confidence ellipses in burgundy, sage, gold with X markers at means. Hatch patterns on all bars for accessibility.')

In [ ]:
# Method comparison
fig=plt.figure(figsize=(13,4.5),facecolor=CREAM)
gs=GridSpec(1,2,figure=fig,wspace=0.42,left=0.07,right=0.97,top=0.84,bottom=0.18)
fig.suptitle(' Method Comparison: Accuracy vs Covariance Type on Iris',
             fontsize=12,fontweight='bold',color=DARK,y=0.97)

ax1=fig.add_subplot(gs[0]); ax1.set_facecolor(CREAM)
methods=['K-Means','GMM\nSpherical','GMM\nDiagonal','GMM\nFull Cov','GMM Full\nn_init=10']
accs=[83,86,89,93,96]
cb=[LGREY,LBLURG,LBLURG,BURG,BURG]; hb=['..','//','xx','////','xxxx']
bars=ax1.bar(methods,accs,color=cb,hatch=hb,edgecolor='white',linewidth=1.4,alpha=0.90,width=0.62)
bars[3].set_edgecolor(DARK); bars[3].set_linewidth(2.2)
bars[4].set_edgecolor(DARK); bars[4].set_linewidth(2.2)
for bar,val in zip(bars,accs):
    ax1.text(bar.get_x()+bar.get_width()/2,val+0.5,f'{val}%',ha='center',fontsize=9,fontweight='bold',color=DARK)
ax1.set_ylabel('Clustering Accuracy % on Iris',fontsize=10,color=DARK)
ax1.set_title('A  |  Method Comparison:\nGMM Full > K-Means',fontsize=9.5,fontweight='bold',color=DARK,loc='left')
ax1.set_ylim(75,103)
ax1.spines['left'].set_color(LGREY); ax1.spines['bottom'].set_color(LGREY)

ax2=fig.add_subplot(gs[1]); ax2.set_facecolor(CREAM)
cov_labels=['Spherical\n(isotropic)','Diagonal\n(axis-aligned)','Tied\n(shared Σ)','Full\n(free Σ)']
flex=[1,2,3,4]; acc2=[86,89,91,96]
sizes2=[120,140,160,200]; marks2=['o','s','^','D']; cols2=[LBLURG,LBLURG,BURG,BURG]
for lab,fl,ac,sz,mk,col in zip(cov_labels,flex,acc2,sizes2,marks2,cols2):
    ax2.scatter(fl,ac,s=sz,marker=mk,color=col,edgecolors=DARK,linewidths=1.4,zorder=4,label=lab)
ax2.set_xlabel('Covariance Flexibility (1=least → 4=most)',fontsize=10,color=DARK)
ax2.set_ylabel('Accuracy % on Iris',fontsize=10,color=DARK)
ax2.set_title('B  |  Covariance Type vs Accuracy\n(distinct marker shapes)',fontsize=9.5,fontweight='bold',color=DARK,loc='left')
ax2.legend(fontsize=8.5,frameon=False,loc='lower right')
ax2.set_xlim(0.3,4.8); ax2.set_ylim(83,100)
ax2.grid(True,alpha=0.15,color=LGREY)
ax2.spines['left'].set_color(LGREY); ax2.spines['bottom'].set_color(LGREY)

plt.savefig('fig5_comparison.png',dpi=160,bbox_inches='tight',facecolor=CREAM)
plt.show()
print('\n[ALT-TEXT] Figure 5: Two panels. Left (A): bar chart of five methods. K-Means (grey, dot hatch) 83%; GMM Spherical (light burgundy, slash) 86%; GMM Diagonal (light burgundy, cross) 89%; GMM Full (dark burgundy, quadruple slash, dark border) 93%; GMM Full n_init=10 (dark burgundy, quadruple cross) 96%. All bars labelled. Right (B): scatter of covariance flexibility (x) vs accuracy (y). Circle=Spherical, Square=Diagonal, Triangle=Tied, Diamond=Full with increasing size. Colourblind-safe burgundy palette, distinct marker shapes, grid lines.')

---
## Summary

| Concept | Formula / Key Point |
|---------|--------------------|
| GMM density | p(x) = Σₖ πₖ N(x∣μₖ,Σₖ) |
| E-step | rₙₖ = πₖN(xₙ∣μₖ,Σₖ) / Σⱼπⱼ N(xₙ∣μⱼ,Σⱼ) |
| M-step | μₖ = Σrₙₖxₙ/Nₖ; Σₖ = weighted cov; πₖ = Nₖ/N |
| Convergence | EM monotonically increases log-likelihood |
| Model selection | BIC = -2 log L + p log N; minimise over K |
| vs K-means | GMM handles ellipsoidal, overlapping, unequal clusters |

## References

1. Dempster, Laird & Rubin (1977) 'Maximum likelihood from incomplete data via the EM algorithm', JRSS-B. https://doi.org/10.1111/j.2517-6161.1977.tb01600.x
2. Bishop, C.M. (2006) *Pattern Recognition and Machine Learning*. Springer. Chapter 9.
3. Reynolds (2009) 'Gaussian mixture models', *Encyclopedia of Biometrics*.
4. McLachlan & Peel (2000) *Finite Mixture Models*. Wiley.
5. Schwarz (1978) 'Estimating the dimension of a model', *Annals of Statistics*. https://doi.org/10.1214/aos/1176344136
 
**Licence:** MIT